In [1]:
import pandas as pd
import os
from os.path import dirname


root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/


In [2]:
dataset = "sepsis_cases_1"

In [3]:
if(dataset == "BPI12_DECLINED_COMPLETE"):
    raw_data = pd.read_csv(f"datasets/original/{dataset}.csv")
else:
    raw_data = pd.read_csv(f"datasets/original/{dataset}.csv",sep=";")

raw_data.head()


,Diagnose,DiagnosticArtAstrup,DiagnosticBlood,DiagnosticECG,DiagnosticIC,DiagnosticLacticAcid,DiagnosticLiquor,DiagnosticOther,DiagnosticSputum,DiagnosticUrinaryCulture,DiagnosticUrinarySediment,DiagnosticXthorax,DisfuncOrg,Hypotensie,Hypoxie,InfectionSuspected,Infusion,Oligurie,SIRSCritHeartRate,SIRSCritLeucos,SIRSCritTachypnea,SIRSCritTemperature,SIRSCriteria2OrMore,Age,Case ID,Activity,org:group,CRP,LacticAcid,Leucocytes,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases,label
0,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Registration,A,0.0,0.0,0.0,2014-10-22 09:15:41,555,10,2,9,0.000000,0.000000,1,81,regular
1,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,Leucocytes,B,0.0,0.0,9.6,2014-10-22 09:27:00,567,10,2,9,0.000000,11.316667,2,81,regular
2,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,CRP,B,21.0,0.0,9.6,2014-10-22 09:27:00,567,10,2,9,0.000000,11.316667,3,81,regular
3,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,LacticAcid,B,21.0,2.2,9.6,2014-10-22 09:27:00,567,10,2,9,11.316667,11.316667,4,81,regular
4,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Triage,C,21.0,2.2,9.6,2014-10-22 09:33:37,573,10,2,9,6.616667,17.933333,5,81,regular


In [4]:
if dataset=="BPI12_DECLINED_COMPLETE":
    tab_all = raw_data.rename(
        columns={"case:concept:name": "CaseID", "concept:name": "Activity"}
    )
else:
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID", "concept:name": "Activity"}
    )



In [5]:
from datetime import datetime
import time

def translate_time(time_str):
    return datetime.fromisoformat(time_str).timestamp()

In [6]:
tab_all["time:timestamp"] = tab_all["time:timestamp"].apply(translate_time)

In [7]:
#To predict remaining cycle time we will add a new column.

#compute final timestamp for each trace.
tab_all["case_end_ts"] = (
    tab_all
    .groupby("CaseID")["time:timestamp"]
    .transform("max")
)

# add column remaining_time in seconds (float)
tab_all["remaining_time"] = (
    tab_all["case_end_ts"] - tab_all["time:timestamp"]
)

#clean up
tab_all.drop(columns=["case_end_ts"], inplace=True)

In [8]:
tab_all.head()

,Diagnose,DiagnosticArtAstrup,DiagnosticBlood,DiagnosticECG,DiagnosticIC,DiagnosticLacticAcid,DiagnosticLiquor,DiagnosticOther,DiagnosticSputum,DiagnosticUrinaryCulture,DiagnosticUrinarySediment,DiagnosticXthorax,DisfuncOrg,Hypotensie,Hypoxie,InfectionSuspected,Infusion,Oligurie,SIRSCritHeartRate,SIRSCritLeucos,SIRSCritTachypnea,SIRSCritTemperature,SIRSCriteria2OrMore,Age,CaseID,Activity,org:group,CRP,LacticAcid,Leucocytes,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases,label,remaining_time
0,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Registration,A,0.0,0.0,0.0,1.413955e+09,555,10,2,9,0.000000,0.000000,1,81,regular,968359.0
1,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,Leucocytes,B,0.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,2,81,regular,967680.0
2,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,CRP,B,21.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,3,81,regular,967680.0
3,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,LacticAcid,B,21.0,2.2,9.6,1.413956e+09,567,10,2,9,11.316667,11.316667,4,81,regular,967680.0
4,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Triage,C,21.0,2.2,9.6,1.413956e+09,573,10,2,9,6.616667,17.933333,5,81,regular,967283.0


In [9]:
split_ratio = 2 / 3

first_act_tab = (
    tab_all.groupby("CaseID").first().sort_values("time:timestamp").reset_index()
)
first_act_tab = first_act_tab[
    ~first_act_tab.duplicated(subset=["CaseID", "Activity"], keep="first")
]
first_act_tab = first_act_tab.reset_index(drop=True)

list_train_valid_cases = list(
    first_act_tab[: int(split_ratio * len(first_act_tab))]["CaseID"].unique()
)

list_train_cases = list_train_valid_cases[: int(len(list_train_valid_cases) * 0.8)]
tab_train = tab_all[tab_all["CaseID"].isin(list_train_cases)].reset_index(drop=True)

list_valid_cases = list_train_valid_cases[int(len(list_train_valid_cases) * 0.8) :]
tab_valid = tab_all[tab_all["CaseID"].isin(list_valid_cases)].reset_index(drop=True)

list_test_cases = list(
    first_act_tab[int(split_ratio * len(first_act_tab)) :]["CaseID"].unique()
)
tab_test = tab_all[tab_all["CaseID"].isin(list_test_cases)].reset_index(drop=True)

In [10]:
tab_all.to_csv(data_dir_processed + f"{dataset}_processed_all.csv", index=False)

In [11]:
tab_train.to_csv(data_dir_processed+ f"{dataset}_processed_train.csv", index = False)

In [12]:
tab_valid.to_csv(data_dir_processed+f"{dataset}_processed_valid.csv", index = False)

In [13]:
tab_test.to_csv(data_dir_processed+ f"{dataset}_processed_test.csv", index = False)